# LangChain Length-Based Example Selector Reference

Developer-facing statements defined in `langchain_core.example_selectors.length_based`.

# `LengthBasedExampleSelector: BaseExampleSelector, BaseModel`

Selects prompt examples in their stored order while their formatted lengths fit within a configured maximum length.

## Fields

```python
examples: list[dict[str, Any]] # Examples expected by the prompt template
example_prompt: PromptTemplate # Prompt template used to format each example
get_text_length: Callable[[str], int] = _get_length_based # Function used to measure text length
max_length: int = 2048 # Maximum combined input and example length
example_text_lengths: list[int] = Field(default_factory=list) # Precomputed length of each formatted example
```

The default length function returns:

```python
len(re.split(r"\n| ", text))
```

## Constructor

```python
LengthBasedExampleSelector(
    *,
    examples: list[dict[str, Any]], # Examples expected by the prompt template
    example_prompt: PromptTemplate, # Template used to format each example
    get_text_length: Callable[[str], int] = _get_length_based, # Text-length function
    max_length: int = 2048, # Maximum combined input and example length
    example_text_lengths: list[int] = Field(default_factory=list), # Optional precomputed example lengths
) -> None
```

## Methods

### `add_example`

Appends an example and stores the length of its formatted text.

```python
add_example(
    self,
    example: dict[str, str], # Example values keyed by prompt input-variable name
) -> None
```

### `aadd_example`

Asynchronously adds an example by directly calling `add_example()`.

```python
async aadd_example(
    self,
    example: dict[str, str], # Example values keyed by prompt input-variable name
) -> None
```

This override does not use the executor-backed wrapper inherited from `BaseExampleSelector`.

### `post_init`

Populates `example_text_lengths` after model validation when no lengths were supplied.

```python
@model_validator(mode="after")
post_init(
    self,
) -> Self # Validated selector
```

Each example is formatted with `example_prompt`, and its length is calculated with `get_text_length`. When `example_text_lengths` is already non-empty, the supplied values are kept unchanged.

### `select_examples`

Selects examples that fit within the remaining length budget.

```python
select_examples(
    self,
    input_variables: dict[str, str], # Current prompt input values
) -> list[dict[str, Any]] # Selected examples
```

The input values are joined with spaces and measured using `get_text_length`. Their length is subtracted from `max_length`.

Examples are then considered from the beginning of `examples`. An example is included when its precomputed length fits within the remaining budget. Selection stops immediately when the next example does not fit; later examples are not considered.

### `aselect_examples`

Asynchronously selects examples by directly calling `select_examples()`.

```python
async aselect_examples(
    self,
    input_variables: dict[str, str], # Current prompt input values
) -> list[dict[str, Any]] # Selected examples
```

This override does not use the executor-backed wrapper inherited from `BaseExampleSelector`.

In [ ]:
from langchain_core.example_selectors import LengthBasedExampleSelector # Import the selector
from langchain_core.prompts import PromptTemplate # Import the prompt template

examples = [ # Create example input-output pairs
    {"input": "happy", "output": "sad"}, # First example
    {"input": "tall", "output": "short"}, # Second example
    {"input": "fast", "output": "slow"}, # Third example
] # Finish the example list

example_prompt = PromptTemplate( # Define how each example will be formatted
    input_variables=["input", "output"], # Define the required variables
    template="Input: {input}\nOutput: {output}", # Define the example format
) # Finish creating the template

selector = LengthBasedExampleSelector( # Create the length-based selector
    examples=examples, # Provide the available examples
    example_prompt=example_prompt, # Provide the formatting template
    max_length=10, # Set the maximum combined length
) # Finish creating the selector

selected_examples = selector.select_examples( # Select examples synchronously
    {"input": "large"} # Provide the current user input
) # Finish selecting examples

print("Example lengths:", selector.example_text_lengths) # Display each formatted example length
print("Selected examples:") # Display a heading

for example in selected_examples: # Visit every selected example
    print(example) # Display the example

await selector.aadd_example( # Add another example asynchronously in Jupyter
    {"input": "hot", "output": "cold"} # Provide the new example
) # Finish adding the example

async_examples = await selector.aselect_examples( # Select examples asynchronously
    {"input": "small"} # Provide another user input
) # Finish asynchronous selection

print("\nTotal examples:", len(selector.examples)) # Display the updated example count
print("Asynchronously selected examples:", async_examples) # Display the selected examples